# A&R Analyst Portfolio Project — Part 2
### Live Data: Emerging Artist Momentum Signals

**Prepared by:** Manav Desai

Part 1 established that static audio characteristics have become weaker predictors of hit potential over time. This section shifts to momentum-based signals — using live Spotify search and Last.fm listener/engagement data — to identify artists showing real traction, as a more current approach to scouting.

In [33]:
# Authenticate with Spotify using Client Credentials flow (works for public data, no user login needed)
import os
from dotenv import load_dotenv
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

# Load API keys from .env file
load_dotenv()

client_id = os.getenv("SPOTIFY_CLIENT_ID")
client_secret = os.getenv("SPOTIFY_CLIENT_SECRET")

# Authenticate with Spotify using Client Credentials flow
# (this flow works for accessing public data — no user login needed)
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=client_id,
    client_secret=client_secret
))

print("Spotify connection established")



Spotify connection established


## 1. Testing API Access

Before building the full pipeline, confirming which Spotify endpoints are still accessible for a standard developer app (given the access restrictions discussed below).

In [6]:
# Test basic search functionality (this endpoint should NOT be affected by the Nov 2024 deprecations)
results = sp.search(q='year:2026', type='track', limit=10)
for item in results['tracks']['items']:
    print(item['name'], '-', item['artists'][0]['name'])

hate that i made you love me - Ariana Grande
Janice STFU - Drake
I Knew It, I Knew You - From "Toy Story 5" - Taylor Swift
stupid song - Olivia Rodrigo
Choosin' Texas - Ella Langley


In [10]:
# I'm grabbing the first track from my earlier search results to test what artist-level data is accessible
first_track = results['tracks']['items'][0]
first_artist_id = first_track['artists'][0]['id']
first_artist_name = first_track['artists'][0]['name']

print(f"Testing artist lookup for: {first_artist_name}")

# Pulling the artist's full profile using their Spotify ID
artist_details = sp.artist(first_artist_id)

# Checking exactly which fields are available (expecting popularity, followers, genres)
print("Available artist fields:", list(artist_details.keys()))

# Checking whether track-level popularity is available as an alternative
print("Available track fields:", list(first_track.keys()))
print("Track popularity value:", first_track.get('popularity'))

Testing artist lookup for: Ariana Grande
Available artist fields: ['external_urls', 'href', 'id', 'images', 'name', 'type', 'uri']
Available track fields: ['album', 'artists', 'disc_number', 'duration_ms', 'explicit', 'external_ids', 'external_urls', 'href', 'id', 'is_local', 'is_playable', 'name', 'track_number', 'type', 'uri']
Track popularity value: None


**Finding:** As of 2026, Spotify's Developer API restricts `popularity`, `followers`, and `genres` fields for standard developer-access apps — both at the artist and track level. This reflects policy changes Spotify rolled out in November 2024 and expanded in February 2026, restricting these fields to apps with Extended Access (which requires 250,000+ monthly active users).

**Adaptation:** Since these fields aren't available via Spotify's API, this project uses Spotify's search/catalog endpoints (still accessible) for track discovery and metadata, combined with the Last.fm API for momentum signals (listener counts, play counts, tags) as a substitute data source.


## 2. Last.fm Integration (Momentum Signals)

Since Spotify's API restricts popularity/follower/genre data for standard developer access (see finding above), this project uses Last.fm's API as a substitute source for artist momentum signals — specifically listener counts and play counts.

In [34]:
from dotenv import load_dotenv
import os

# Reloading .env with override=True since the Last.fm key was added after the notebook's initial load_dotenv() call
load_dotenv(override=True)

lastfm_api_key = os.getenv("LASTFM_API_KEY")

# Confirm the key loaded without printing the actual secret value
print("API key loaded:", lastfm_api_key is not None)

API key loaded: True


In [35]:
# Testing the Last.fm connection with a known artist
params = {
    "method": "artist.getInfo",
    "artist": "Ariana Grande",
    "api_key": lastfm_api_key,
    "format": "json"
}

response = requests.get("http://ws.audioscrobbler.com/2.0/", params=params)
data = response.json()

print(data)

{'artist': {'name': 'Ariana Grande', 'mbid': 'f4fdbb4c-e4b7-47a0-b83b-d91bbfcfa387', 'url': 'https://www.last.fm/music/Ariana+Grande', 'image': [{'#text': 'https://lastfm.freetls.fastly.net/i/u/34s/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'small'}, {'#text': 'https://lastfm.freetls.fastly.net/i/u/64s/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'medium'}, {'#text': 'https://lastfm.freetls.fastly.net/i/u/174s/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'large'}, {'#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'extralarge'}, {'#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': 'mega'}, {'#text': 'https://lastfm.freetls.fastly.net/i/u/300x300/2a96cbd8b46e442fc41c2b86b821562f.png', 'size': ''}], 'streamable': '0', 'ontour': '1', 'stats': {'listeners': '4617096', 'playcount': '1184987084'}, 'similar': {'artist': [{'name': 'Jessie J, Ariana Grande & Nicki Minaj', 'url': 'http

In [36]:
# Confirm the specific fields we need (listener count and play count) are accessible
print("Listeners:", data['artist']['stats']['listeners'])
print("Play count:", data['artist']['stats']['playcount'])

Listeners: 4617096
Play count: 1184987084


## 3. Building the Track Pool

Searching across eight genres to build a varied pool of current tracks, avoiding bias toward any single genre. (Note: Spotify's search endpoint appears to cap results at 10 per request for standard developer access. The "hip hop" and "alternative" genre tags both returned no results — pop, indie, country, rock, r&b, electronic, and folk were used instead to maintain genre diversity.)

In [45]:
# Searching across multiple genres to build a varied pool of current tracks
search_terms = [
    "genre:pop year:2026",
    "genre:indie year:2026",
    "genre:country year:2026",
    "genre:rock year:2026",
    "genre:r&b year:2026",
    "genre:electronic year:2026",
    "genre:folk year:2026",
    "genre:alternative year:2026"
]

all_tracks = []

for term in search_terms:
    results = sp.search(q=term, type='track', limit=10, market='US')
    tracks = results['tracks']['items']
    all_tracks.extend(tracks)
    print(f"'{term}' returned {len(tracks)} tracks")

print(f"\nTotal tracks collected: {len(all_tracks)}")

'genre:pop year:2026' returned 10 tracks
'genre:indie year:2026' returned 10 tracks
'genre:country year:2026' returned 10 tracks
'genre:rock year:2026' returned 10 tracks
'genre:r&b year:2026' returned 10 tracks
'genre:electronic year:2026' returned 10 tracks
'genre:folk year:2026' returned 10 tracks
'genre:alternative year:2026' returned 0 tracks

Total tracks collected: 70


## 4. Enriching with Momentum Data

Looking up each unique artist's Last.fm listener and play counts.

In [46]:
# Extract unique artist names from our track pool (avoiding duplicate Last.fm lookups)
artist_names = list(set(track['artists'][0]['name'] for track in all_tracks))
print(f"Found {len(artist_names)} unique artists")
print(artist_names)

Found 33 unique artists
['kwn', 'Kehlani', 'Bella Kay', 'Bryson Tiller', 'Avery Anna', 'HUGEL', 'pupsies', 'Josh Fawaz', 'Ariana Grande', 'Malcolm Todd', 'Steve Lacy', 'Beyoncé', 'Cameron Whitcomb', 'Noah Kahan', 'Tucker Wetmore', 'Dexter and The Moonrocks', 'Brent Faiyaz', 'beabadoobee', 'Ella Langley', 'mgk', 'PIXY', 'Rainbow Kitten Surprise', 'jigitz', 'Olivia Rodrigo', 'D A N N Y', 'Oliver Tree', 'Motionless In White', 'Tame Impala', 'F3miii', 'Ian Asher', 'Don Toliver', 'Maphra', 'honestav']


In [47]:
import time

# Collect Last.fm listener/playcount stats for each unique artist
artist_stats = []

for name in artist_names:
    params = {
        "method": "artist.getInfo",
        "artist": name,
        "api_key": lastfm_api_key,
        "format": "json"
    }
    
    response = requests.get("http://ws.audioscrobbler.com/2.0/", params=params)
    data = response.json()
    
    # Some smaller/newer artists might not exist in Last.fm's database at all
    if 'artist' in data:
        listeners = int(data['artist']['stats']['listeners'])
        playcount = int(data['artist']['stats']['playcount'])
        artist_stats.append({'artist': name, 'listeners': listeners, 'playcount': playcount})
        print(f"{name}: {listeners:,} listeners, {playcount:,} plays")
    else:
        print(f"{name}: not found on Last.fm")
    
    # Being polite to Last.fm's API — avoid sending requests too fast
    time.sleep(0.5)
    

kwn: 243,771 listeners, 6,336,419 plays
Kehlani: 1,792,038 listeners, 76,144,100 plays
Bella Kay: 438,705 listeners, 11,318,081 plays
Bryson Tiller: 1,761,284 listeners, 111,020,928 plays
Avery Anna: 95,543 listeners, 1,454,104 plays
HUGEL: 675,559 listeners, 8,744,398 plays
pupsies: 144,875 listeners, 3,441,606 plays
Josh Fawaz: 22,021 listeners, 146,645 plays
Ariana Grande: 4,617,096 listeners, 1,184,987,084 plays
Malcolm Todd: 1,646,529 listeners, 135,418,257 plays
Steve Lacy: 3,237,021 listeners, 303,323,626 plays
Beyoncé: 6,494,338 listeners, 718,545,462 plays
Cameron Whitcomb: 176,172 listeners, 4,234,551 plays
Noah Kahan: 1,681,400 listeners, 215,010,280 plays
Tucker Wetmore: 175,103 listeners, 3,542,182 plays
Dexter and The Moonrocks: 332,222 listeners, 4,807,974 plays
Brent Faiyaz: 2,120,169 listeners, 247,369,592 plays
beabadoobee: 2,747,381 listeners, 345,356,925 plays
Ella Langley: 484,532 listeners, 13,002,579 plays
mgk: 1,091,813 listeners, 43,253,080 plays
PIXY: 277,208 

In [48]:
# Convert results into a DataFrame, sorted by listener count for easy comparison
artist_df = pd.DataFrame(artist_stats)
artist_df = artist_df.sort_values('listeners', ascending=False).reset_index(drop=True)
display(artist_df)

,artist,listeners,playcount
0,Beyoncé,6494338,718545462
1,Ariana Grande,4617096,1184987084
2,Tame Impala,4553333,442949472
3,Olivia Rodrigo,3287911,672410238
4,Steve Lacy,3237021,303323626
5,beabadoobee,2747381,345356925
6,Don Toliver,2645569,300661138
7,Brent Faiyaz,2120169,247369592
8,Kehlani,1792038,76144100
9,Bryson Tiller,1761284,111020928


## 5. Classifying Artists by Momentum Tier

Using natural breaks in the listener-count distribution to classify artists into three tiers: Established, Emerging, and Early-Stage.

In [49]:
def classify_artist(listeners):
    if listeners >= 1_000_000:
        return 'Established'
    elif listeners >= 100_000:
        return 'Emerging'
    else:
        return 'Early-Stage'

artist_df['tier'] = artist_df['listeners'].apply(classify_artist)
display(artist_df)


,artist,listeners,playcount,tier
0,Beyoncé,6494338,718545462,Established
1,Ariana Grande,4617096,1184987084,Established
2,Tame Impala,4553333,442949472,Established
3,Olivia Rodrigo,3287911,672410238,Established
4,Steve Lacy,3237021,303323626,Established
5,beabadoobee,2747381,345356925,Established
6,Don Toliver,2645569,300661138,Established
7,Brent Faiyaz,2120169,247369592,Established
8,Kehlani,1792038,76144100,Established
9,Bryson Tiller,1761284,111020928,Established


## 6. Engagement Ratio: A Second Momentum Signal

Raw listener count alone doesn't capture fan loyalty. Calculating plays-per-listener surfaces artists with unusually devoted audiences, regardless of tier.

In [50]:
# Calculate plays-per-listener as an engagement/loyalty signal, independent of raw listener count
artist_df['plays_per_listener'] = (artist_df['playcount'] / artist_df['listeners']).round(1)
display(artist_df.sort_values('plays_per_listener', ascending=False))

,artist,listeners,playcount,tier,plays_per_listener
1,Ariana Grande,4617096,1184987084,Established,256.7
3,Olivia Rodrigo,3287911,672410238,Established,204.5
10,Noah Kahan,1681400,215010280,Established,127.9
5,beabadoobee,2747381,345356925,Established,125.7
7,Brent Faiyaz,2120169,247369592,Established,116.7
6,Don Toliver,2645569,300661138,Established,113.6
0,Beyoncé,6494338,718545462,Established,110.6
2,Tame Impala,4553333,442949472,Established,97.3
4,Steve Lacy,3237021,303323626,Established,93.7
12,Malcolm Todd,1646529,135418257,Established,82.2


In [51]:
# Save the enriched artist dataset so Part 3 can use it without re-running the API pipeline
artist_df.to_csv('data/emerging_artists.csv', index=False)
print("Saved to data/emerging_artists.csv")

Saved to data/emerging_artists.csv


**Takeaway:** Motionless In White (Emerging tier, 948K listeners) shows a plays-per-listener ratio of 68.8 — the highest in the entire dataset, exceeding even Established artists like Rainbow Kitten Surprise (41.0) and mgk (39.6). This suggests an unusually loyal, highly-engaged fanbase relative to audience size — a potential signal of untapped growth potential ahead of wider audience exposure.

## Summary & Next Steps

This section demonstrates a momentum-based scouting approach, built to work around Spotify's 2024–2026 API access restrictions on popularity/follower/genre data. Combining Spotify search (track discovery across 8 genres) with Last.fm (listener/engagement data) surfaces a data-driven shortlist of 33 unique artists — 15 Established, 13 Emerging, 5 Early-Stage — worth scouting, beyond what raw popularity alone would suggest.

**Next steps:** Develop a combined scoring model integrating these momentum signals with the historical hit-characteristics findings from Part 1 (see Part 3).